# Lab 03: Causal Multi-Head Self-Attention & LayerNorm from Scratch

**Companion to [Module 05](../curriculum/03-modern-ai-and-genai/05_nlp_cv_and_multimodal.md) & [Local Project 3](../projects/README.md)** — Build the mathematical heart of GPT / Llama / Gemini in pure NumPy.

In [ ]:
import numpy as np

def multi_head_causal_attention(
    x: np.ndarray,
    w_q: np.ndarray,
    w_k: np.ndarray,
    w_v: np.ndarray,
    w_o: np.ndarray,
    num_heads: int = 2,
) -> np.ndarray:
    batch, seq_len, d_model = x.shape
    head_dim = d_model // num_heads

    def split_heads(t: np.ndarray) -> np.ndarray:
        # (B, T, D) -> (B, num_heads, T, head_dim)
        return t.reshape(batch, seq_len, num_heads, head_dim).transpose(0, 2, 1, 3)

    q = split_heads(x @ w_q)
    k = split_heads(x @ w_k)
    v = split_heads(x @ w_v)

    scores = (q @ k.transpose(0, 1, 3, 2)) / np.sqrt(head_dim)
    mask = np.triu(np.ones((seq_len, seq_len), dtype=bool), k=1)
    scores = np.where(mask, -1e9, scores)

    shifted = scores - np.max(scores, axis=-1, keepdims=True)
    attn = np.exp(shifted) / np.sum(np.exp(shifted), axis=-1, keepdims=True)

    merged = (attn @ v).transpose(0, 2, 1, 3).reshape(batch, seq_len, d_model)
    return merged @ w_o

rng = np.random.default_rng(0)
d = 16
wq, wk, wv, wo = (rng.normal(scale=0.1, size=(d, d)) for _ in range(4))
tokens_a = rng.normal(size=(2, 5, d))
tokens_b = tokens_a.copy()
tokens_b[:, 4, :] *= -500.0  # Mutate final token t=4

out_a = multi_head_causal_attention(tokens_a, wq, wk, wv, wo, num_heads=4)
out_b = multi_head_causal_attention(tokens_b, wq, wk, wv, wo, num_heads=4)

assert out_a.shape == (2, 5, d)
assert np.allclose(out_a[:, :4, :], out_b[:, :4, :]), 'Future token leaked into earlier positions!'
print('Multi-Head Causal Self-Attention verified! Output shape:', out_a.shape)